# Colab 4
## Collaborative filtering

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyspark
from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark import SparkContext, SparkConf
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS

In [2]:
# create the session
conf = SparkConf().set("spark.ui.port", "4050")

In [3]:
# create the context
sc = pyspark.SparkContext(conf=conf)
spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/03 23:49:21 WARN Utils: Your hostname, panteliss-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.255.221.4 instead (on interface en0)
26/02/03 23:49:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/03 23:49:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
schema_ratings = StructType([
    StructField("user_id", IntegerType(), False),
    StructField("item_id", IntegerType(), False),
    StructField("rating", IntegerType(), False),
    StructField("timestamp", IntegerType(), False)])

In [5]:
schema_items = StructType([
    StructField("item_id", IntegerType(), False),
    StructField("movie", StringType(), False)])

In [6]:
training = spark.read.option("sep", "\t").csv("../../assets/datasets/MovieLens.training", header=False, schema=schema_ratings)
test = spark.read.option("sep", "\t").csv("../../assets/datasets/MovieLens.test", header=False, schema=schema_ratings)
items = spark.read.option("sep", "|").csv("../../assets/datasets/MovieLens.item", header=False, schema=schema_items)

In [7]:
training.printSchema()
test.printSchema()
items.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- rating: integer (nullable = true)
 |-- timestamp: integer (nullable = true)

root
 |-- user_id: integer (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- rating: integer (nullable = true)
 |-- timestamp: integer (nullable = true)

root
 |-- item_id: integer (nullable = true)
 |-- movie: string (nullable = true)



### Getting the size of the train and test sets

In [8]:
print(f"Training set size: {training.count()}")
print(f"Test set size: {test.count()}")
print(items.count())

Training set size: 80000
Test set size: 20000
1682


### Training a model with the Alternative least squares method

In [10]:
training.cache()
training.count()
als = ALS(maxIter = 5,userCol = "user_id", itemCol = "item_id", ratingCol="rating", coldStartStrategy="drop")
model = als.fit(training)

26/02/03 23:50:17 WARN CacheManager: Asked to cache already cached data.
26/02/03 23:50:20 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/02/03 23:50:21 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


### Evaluate the model using RMSE in test dataset

In [13]:
predictions = model.transform(test)
evaluator = RegressionEvaluator(metricName="rmse", labelCol = "rating", predictionCol = "prediction")
rmse = evaluator.evaluate(predictions)
print("Root-mean-square error = " + str(rmse))

Root-mean-square error = 0.9430515312112326


### Get the top-10 recommendations for each user

In [ ]:
userRecs = model.recommendForAllUsers(10)
userRecs.printSchema()

root
 |-- user_id: integer (nullable = false)
 |-- recommendations: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- item_id: integer (nullable = true)
 |    |    |-- rating: float (nullable = true)



In [20]:
userRecc = userRecs.select("user_id", explode("recommendations").alias("recommend")).select("user_id", col("recommend.item_id"), col("recommend.rating"))

In [25]:
totalRec = userRecc.join(items, on="item_id", how="inner")
totalRec.where(totalRec.item_id == 50).show()

+-------+-------+---------+----------------+
|item_id|user_id|   rating|           movie|
+-------+-------+---------+----------------+
|     50|      5|4.5742583|Star Wars (1977)|
|     50|      9|5.0850315|Star Wars (1977)|
|     50|     16| 5.384225|Star Wars (1977)|
|     50|     22| 4.605069|Star Wars (1977)|
|     50|     28| 4.777958|Star Wars (1977)|
|     50|     37|4.3894978|Star Wars (1977)|
|     50|     44|4.5734024|Star Wars (1977)|
|     50|     57|4.5420995|Star Wars (1977)|
|     50|     64|4.3687496|Star Wars (1977)|
|     50|     78| 4.568984|Star Wars (1977)|
|     50|     91|4.5206094|Star Wars (1977)|
|     50|     96|4.7739124|Star Wars (1977)|
|     50|    117|4.7613945|Star Wars (1977)|
|     50|    137| 4.859361|Star Wars (1977)|
|     50|    148|4.6768246|Star Wars (1977)|
|     50|    157| 4.798106|Star Wars (1977)|
|     50|    163|3.7716477|Star Wars (1977)|
|     50|    175| 4.374239|Star Wars (1977)|
|     50|    178|4.6160808|Star Wars (1977)|
|     50| 